# Does a one-macro horizon improve raw-MSE trajectories?

Use a GPU runtime. Run raw-MSE planning with **planning_horizon=1** and **actions_per_plan=1**. Each plan predicts and executes up to five simulator actions, then replans from the latest observation. The original baseline planned and executed up to five macros per call.

The 36 narrow-square Band 1 starts, goals, seeds, model weights, CEM population and iterations, action cap, and episode action budget stay matched. This notebook collects and audits trajectories before any adapter fitting. Results go to a separate Drive directory.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
SOURCE_RUN=DRIVE/'datasets/curriculum_cost_adapter/radial_multigoal_ranking_v1/20261106T195304628603Z'
assert (SOURCE_RUN/'branches/multi/complete.json').is_file(), 'Set SOURCE_RUN to your completed multi-goal ranking run.'
BASELINE_RUN=DRIVE/'datasets/curriculum_cost_adapter/experience_temporal_half_square_v1/20261107T150425413490Z'
RESUME_RUN_ID=None  # Set to the printed ID to resume an interrupted run.
STARTS_PER_GOAL=6
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/raw_mse_half_square_h1_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-raw-mse-half-square-h1')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='f2c3e6127571a29b8e8072dfc4ce6526a824b1b8'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/raw-mse-half-square-h1-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Frozen ranking source:',SOURCE_RUN)
print('Run ID:',RUN_ID)
print('New raw-MSE test results:',OUTPUT)

## Load the pinned code and frozen models

The world-model checkpoint, encoder and renderer are the same ones used in the previous planning comparison.

If Drive interrupts a checksum read, the code retries and prints the file path. For a persistent error, remount Drive and rerun the affected band.

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

## Run the one-macro raw-MSE test

Set `RESUME_RUN_ID` in the setup cell to reuse completed cases from this run. The saved baseline is read in the comparison cells.

In [ ]:
started=time.monotonic()
args=[sys.executable,'-m','src.mylonas_ijepa_wm.experience_curriculum','collect-raw',
      '--source-run',SOURCE_RUN,'--output-dir',OUTPUT,'--world-model-checkpoint',CHECKPOINT,
      '--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda',
      '--starts-per-goal',STARTS_PER_GOAL,'--labels','temporal','--sampling','half-square',
      '--planning-horizon',1,'--actions-per-plan',1]
shell(args,cwd=REPO)
print('Collection minutes:',round((time.monotonic()-started)/60,2))

## Compare matched raw-MSE trajectories

Report success on all starts. Compare trajectory quality on the starts both planners solve. Extra travel is the distance traveled beyond a straight line to the actual endpoint. An action counts as toward the goal if physical distance decreases by more than 0.00001 units. Physical distance is used for this audit.

In [ ]:
import io
import numpy as np
import pandas as pd

def read_json(path):
    return json.loads(path.read_text())

def load_band1(root):
    experiment=read_json(root/'experiment.json')
    manifest=experiment['manifest']
    cases=manifest['stages'][0]
    stage=read_json(root/'stages/band_1.json')
    expected=hashlib.sha256(json.dumps(cases,sort_keys=True,allow_nan=False).encode()).hexdigest()
    if (stage['status']!='complete' or stage['condition']['collector']!='raw'
            or stage['cases_digest']!=expected or stage['episodes']!=len(cases)):
        raise ValueError(f'Incomplete or mismatched raw-MSE Band 1 in {root}')
    runtime=read_json(root/'runtime.json')
    if runtime['runtime_id']!=stage['runtime_id']:
        raise ValueError(f'Band 1 runtime differs in {root}')
    rows=[]
    for case in cases:
        ident=case['case_id']
        record=read_json(root/f'episodes/band_1/{ident}.json')
        if (record['case']!=case or record['runtime_id']!=stage['runtime_id']
                or record['condition']!=stage['condition']):
            raise ValueError(f'Episode provenance differs for {ident}')
        path=(root/record['raw_path']).resolve()
        path.relative_to(root.resolve())
        data=path.read_bytes()
        if hashlib.sha256(data).hexdigest()!=record['raw_sha256']:
            raise ValueError(f'Raw checksum differs for {ident}')
        raw=torch.load(io.BytesIO(data),map_location='cpu',weights_only=True)
        xy=raw['states'][:,:2].numpy().astype(float)
        if (raw['case']!=case or raw['runtime_id']!=stage['runtime_id'] or raw['success']!=record['success']
                or len(xy)!=record['micro_steps']+1 or not np.isfinite(xy).all()
                or not np.array_equal(raw['primitive_steps'].numpy(),np.arange(len(xy)))):
            raise ValueError(f'Raw episode differs for {ident}')
        rows.append(dict(case=ident,goal=tuple(case['goal']),success=record['success'],xy=xy,
                         planner_calls=record['mpc_updates']))
    return experiment,runtime['signature'],rows

baseline,old_sig,old_rows=load_band1(BASELINE_RUN)
current,new_sig,new_rows=load_band1(OUTPUT)
old_manifest=baseline['manifest']
new_manifest=current['manifest']
expected=dict(old_manifest,planning=dict(old_manifest['planning'],planning_horizon=1,actions_per_plan=1))
if (old_manifest['planning']['actions_per_plan']!=5 or old_manifest['planning']['planning_horizon']!=5
        or new_manifest!=expected):
    raise ValueError('The manifests differ beyond the requested horizon and execution stride.')
for key in ('handcrafted','fitting','scale'):
    if baseline[key]!=current[key]:
        raise ValueError(f'Experiment settings differ for {key}.')
for key in ('models','dino_commit','dino_source','precision','encoder_batch_size'):
    if old_sig[key]!=new_sig[key]:
        raise ValueError(f'Runtime settings differ for {key}.')
for label,sig in [('5-macro horizon',old_sig),('1-macro horizon',new_sig)]:
    variant=sig['runtime_variant']
    print(label,'· GPU:',variant['device_name'],'· Torch:',sig['torch_version'],
          '· matmul precision:',variant['matmul_precision'])

In [ ]:
MOVE_TOLERANCE=1e-5
SUCCESS_RADIUS=4.5

def measure_path(row):
    xy=row['xy']
    distance=np.linalg.norm(xy-row['goal'],axis=1)
    change=np.diff(distance)
    length=np.linalg.norm(np.diff(xy,axis=0),axis=1).sum()
    displacement=np.linalg.norm(xy[-1]-xy[0])
    return dict(case=row['case'],success=row['success'],actions=len(change),
                initial_distance=distance[0],final_distance=distance[-1],path_length=length,
                extra_travel=max(0.,length-displacement),away_steps=int((change>MOVE_TOLERANCE).sum()),
                away_distance=np.maximum(change,0).sum(),
                toward_steps=int((change<-MOVE_TOLERANCE).sum()),
                toward_pct=100*(change<-MOVE_TOLERANCE).mean() if len(change) else np.nan,
                worst_backtrack=(distance-np.minimum.accumulate(distance)).max())

def choose_three(rows, stats):
    # Use successful rollouts for the visual comparison when at least three exist.
    by_case={r['case']:r for r in rows}
    pool=[s for s in stats if s['success']]
    if len(pool)<3:
        pool=stats
    ordered=sorted(pool,key=lambda s:(s['extra_travel'],s['case']))
    indices=np.linspace(0,len(ordered)-1,min(3,len(ordered)),dtype=int)
    return [by_case[ordered[i]['case']] for i in indices]

runs={'5-macro horizon':old_rows,'1-macro horizon':new_rows}
table=pd.DataFrame([dict(setting=label,goal=row['goal'],planner_calls=row['planner_calls'],**measure_path(row))
                    for label,rows in runs.items() for row in rows])
print('Successes on all matched starts')
display(table.groupby('setting',sort=False)['success'].agg(['sum','count']))
success=table.pivot(index='case',columns='setting',values='success')
both=success.index[success.all(axis=1)]
print('Cases solved only with the one-macro horizon:',success.index[success['1-macro horizon'] & ~success['5-macro horizon']].tolist())
print('Cases solved only with the five-macro horizon:',success.index[success['5-macro horizon'] & ~success['1-macro horizon']].tolist())
print('Both-success pairs:',len(both))
columns=['actions','planner_calls','path_length','extra_travel','away_steps','away_distance','worst_backtrack','toward_pct']
paired=table[table.case.isin(both)]
if len(paired):
    print('Median values on the same successful starts')
    display(paired.groupby('setting',sort=False)[columns].median().round(2))
    wide=paired.pivot(index='case',columns='setting',values=columns)
    delta=wide.xs('1-macro horizon',level=1,axis=1)-wide.xs('5-macro horizon',level=1,axis=1)
    print('Paired deltas · one-macro horizon minus five-macro horizon')
    display(pd.DataFrame({'mean':delta.mean(),'median':delta.median(),
                          'lower_in_H1':(delta < -1e-5).sum(),
                          'higher_in_H1':(delta > 1e-5).sum()}).round(2))
    totals=paired.groupby('setting',sort=False)[['toward_steps','actions']].sum()
    print('Share of executed actions moving toward the goal on shared successes')
    display((100*totals.toward_steps/totals.actions).rename('toward_pct').round(2))
    print('Median extra travel by goal on both-success pairs')
    display(paired.pivot_table(index='goal',columns='setting',values='extra_travel',aggfunc='median').round(2))
table.to_csv(OUTPUT/'band1_paired_detours.csv',index=False)

## The same three paths for each of the two goals

Starts are selected from the baseline by least, middle, and most extra travel among successes. Colors and axes match between horizons.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

goals=sorted({r['goal'] for r in old_rows},key=lambda g:g[1])
fig,axes=plt.subplots(2,2,figsize=(12,11),constrained_layout=True)
new_by_id={r['case']:r for r in new_rows}
for index,goal in enumerate((goals[0],goals[-1])):
    candidates=[r for r in old_rows if r['goal']==goal]
    original=choose_three(candidates,[measure_path(r) for r in candidates])
    replanned=[new_by_id[r['case']] for r in original]
    points=np.vstack([*(r['xy'] for r in [*original,*replanned]),
                      np.asarray(goal)-SUCCESS_RADIUS,np.asarray(goal)+SUCCESS_RADIUS])
    lo,hi=points.min(axis=0)-1,points.max(axis=0)+1
    for ax,(label,rows) in zip(axes[index],[('5-macro horizon',original),('1-macro horizon',replanned)]):
        ax.add_patch(Circle(goal,SUCCESS_RADIUS,fill=False,ls=':',edgecolor='gray'))
        ax.scatter(*goal,marker='*',color='black',s=110,zorder=5)
        for color,r in zip(('tab:blue','tab:orange','tab:green'),rows):
            xy=r['xy']
            stats=measure_path(r)
            tag=f"{r['case']} · {stats['extra_travel']:.1f} extra"
            if not r['success']:
                tag+=' · failed'
            ax.plot(*xy.T,color=color,marker='.',markersize=3,label=tag)
            ax.scatter(*xy[0],s=45,facecolors='none',edgecolors=color)
            ax.scatter(*xy[-1],s=34,marker='x',color=color)
        ax.set(xlim=(lo[0],hi[0]),ylim=(hi[1],lo[1]),aspect='equal',
               xlabel='Ball x',ylabel='Ball y',title=f'{label} · goal {tuple(round(v,2) for v in goal)}')
        ax.legend(fontsize=8)
figure_path=OUTPUT/'band1_paired_detours.png'
fig.savefig(figure_path,dpi=150,bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(figure_path)))